# Notebook C — Automatic Speech Recognition

## Systems compared

| | pretraining | adaptation |
|---|---|---|
| Whisper-small (Bangla) | multilingual, Bangla-adapted checkpoint | LoRA |
| FastConformer hybrid RNNT-CTC (Bangla) | Bengali-specific | full fine-tuning, frozen lower encoder |

Both are evaluated on the same 1,387-utterance held-out test set with the same
scoring function, under both greedy and beam decoding.

## Run order

| session | cells | time |
|---|---|---|
| **1 — Whisper** | C0 → *restart* → C1 → C2 → C3 → C4 | ~9 h |
| **2 — FastConformer** | C0b → *restart* → C1 → C2 → C5 → C6 → C7 | ~5 h |
| **3 — analysis** | C1 → C8 → C8b → C9 → C10 → C11 → C12 | ~25 min |

GPU on throughout. **Quick Save** at the end of every session. Persistence must be **"Files only"**.

C8b must run between C8 and C11: `rescore.py` rebuilds the summary block and discards stored confidence intervals, and `report.py` needs them.


## C0 — environment, Whisper stack (session 1 only)
**Restart & Clear Cell Outputs after this cell**, then continue from C1. Do not
re-run C0 after the restart.

In [ ]:
!pip uninstall -y torchao 2>&1 | tail -1
!pip install -q peft 2>&1 | tail -2
import torch, transformers, peft, platform
print("python", platform.python_version())
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__)
print("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

## C0b — FastConformer stack (session 2 only)

In [ ]:
%%bash
cat > /tmp/con.txt <<'EOF'
numpy==1.26.4
scipy==1.13.1
EOF
pip uninstall -y torchao > /dev/null 2>&1
pip install -q -c /tmp/con.txt "nemo_toolkit[asr]==3.0.0" lightning pytorch-lightning 2>&1 | tail -5
python -c "import numpy, torch; print('numpy', numpy.__version__, '| torch', torch.__version__)"


## C1 — write the scripts

In [1]:
%%writefile /kaggle/working/asr_common.py
"""Shared by every ASR section: text normalisation, WER/CER, result storage."""
import json, re, collections, unicodedata, random
from pathlib import Path

MANIFEST_DIR = Path("/kaggle/working/manifests")
RESULTS = Path("/kaggle/working/results"); RESULTS.mkdir(parents=True, exist_ok=True)
CKPT_DIR = Path("/kaggle/working/ckpt"); CKPT_DIR.mkdir(parents=True, exist_ok=True)
SR = 16000
DIALECTS = ["standard", "chittagong", "mymensingh", "barishal", "sylhet", "noakhali"]

_PUNCT = r"[।,\.\?!;:\-–—_'\"()\[\]{}<>/\\|@#\$%\^&\*\+=~`‌‍]"


def norm(s):
    
    s = unicodedata.normalize("NFC", str(s or "")).replace("<>", " ")
    s = re.sub(_PUNCT, " ", s)
    s = re.sub(r"[0-9০-৯]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def edits(ref, hyp):
    """Levenshtein distance between two sequences."""
    prev = list(range(len(hyp) + 1))
    for i, r in enumerate(ref, 1):
        cur = [i]
        for j, h in enumerate(hyp, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (r != h)))
        prev = cur
    return prev[-1]


def read_manifest(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]


def repetition_rate(hyps, n=3):
    """Fraction of outputs where one n-gram repeats 3+ times in a row.
    Autoregressive decoders collapse this way; CTC structurally cannot."""
    bad = 0
    for h in hyps:
        w = norm(h).split()
        if len(w) < n * 3:
            continue
        grams = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
        run = best = 1
        for a, b in zip(grams, grams[1:]):
            run = run + 1 if a == b else 1
            best = max(best, run)
        bad += best >= 3
    return 100 * bad / max(len(hyps), 1)


def bootstrap_ci(num, den, n_boot=1000, seed=0, alpha=0.05):
    
    rng = random.Random(seed)
    k = len(num)
    if k == 0:
        return (float("nan"), float("nan"))
    vals = []
    for _ in range(n_boot):
        idx = [rng.randrange(k) for _ in range(k)]
        e = sum(num[i] for i in idx)
        d = sum(den[i] for i in idx)
        vals.append(100 * e / max(d, 1))
    vals.sort()
    lo = vals[int(alpha / 2 * n_boot)]
    hi = vals[int((1 - alpha / 2) * n_boot) - 1]
    return (round(lo, 2), round(hi, 2))


def paired_bootstrap(a_num, a_den, b_num, b_den, n_boot=1000, seed=0):
    
    rng = random.Random(seed)
    k = len(a_num)
    wins = 0
    for _ in range(n_boot):
        idx = [rng.randrange(k) for _ in range(k)]
        wa = sum(a_num[i] for i in idx) / max(sum(a_den[i] for i in idx), 1)
        wb = sum(b_num[i] for i in idx) / max(sum(b_den[i] for i in idx), 1)
        wins += (wa < wb)
    p = 1 - wins / n_boot
    return round(p, 4)


def score(rows, hyps, tag, extra=None):
    
    agg = collections.defaultdict(lambda: [0, 0, 0, 0, 0])
    per_clip = []
    for r, hy in zip(rows, hyps):
        rr, hh = norm(r.get("text", "")), norm(hy)
        rw, hw = rr.split(), hh.split()
        e1, n1 = edits(rw, hw), max(len(rw), 1)
        e2, n2 = edits(list(rr), list(hh)), max(len(rr), 1)
        for k in (r.get("dialect", "_"), "ALL"):
            a = agg[k]; a[0] += e1; a[1] += n1; a[2] += e2; a[3] += n2; a[4] += 1
        per_clip.append({"dialect": r.get("dialect", "_"), "ref": r.get("text", ""),
                         "hyp": hy, "wer": round(100 * e1 / n1, 2),
                         "cer": round(100 * e2 / n2, 2),
                         "w_err": e1, "w_ref": n1, "c_err": e2, "c_ref": n2})

    summary = {}
    print(f"\n{tag}")
    print(f"{'dialect':<14}{'n':>6}{'WER':>9}{'95% CI':>16}{'CER':>9}")
    for k in sorted(agg, key=lambda x: (x == "ALL", DIALECTS.index(x) if x in DIALECTS else 99)):
        a = agg[k]
        w, c = 100 * a[0] / max(a[1], 1), 100 * a[2] / max(a[3], 1)
        sub = [r for r in per_clip if k == "ALL" or r["dialect"] == k]
        lo, hi = bootstrap_ci([r["w_err"] for r in sub], [r["w_ref"] for r in sub])
        summary[k] = {"n": a[4], "wer": round(w, 2), "cer": round(c, 2),
                      "wer_ci95": [lo, hi]}
        print(f"{k:<14}{a[4]:>6}{w:>9.2f}{f'[{lo}, {hi}]':>16}{c:>9.2f}")
    rep = repetition_rate(hyps)
    print(f"repetition collapse: {rep:.1f}% of outputs")

    out = {"tag": tag, "summary": summary, "repetition_pct": round(rep, 2),
           "extra": extra or {}, "per_clip": per_clip}
    p = RESULTS / f"{tag}.json"
    json.dump(out, open(p, "w"), ensure_ascii=False, indent=1)
    print("wrote", p)
    return summary


def worst(tag, k=5):
    d = json.load(open(RESULTS / f"{tag}.json"))
    print(f"\nworst {k} clips ({tag}):")
    for r in sorted(d["per_clip"], key=lambda x: -x["wer"])[:k]:
        print(f"  [{r['dialect']}] WER {r['wer']:.0f}")
        print("    REF:", r["ref"][:70])
        print("    HYP:", r["hyp"][:70])


Overwriting /kaggle/working/asr_common.py


In [2]:
%%writefile /kaggle/working/link_a.py
"""Point notebook C at notebook A's saved output.
"""
import json, sys
from pathlib import Path

A_SLUG = "/kaggle/input/notebooks/tasfiaisrat/notebook-a-data-prep"     

DST = Path("/kaggle/working/manifests"); DST.mkdir(parents=True, exist_ok=True)

s = A_SLUG.strip().strip("/")
cands = [Path("/kaggle/input") / s, Path("/" + s)]
ROOT = next((c for c in cands if c.exists()), None)
if ROOT is None:
    print("available under /kaggle/input:")
    for p in sorted(Path("/kaggle/input").rglob("*")):
        if p.is_dir() and (p / "processed").exists():
            print("   ", p)
    sys.exit(f"\ncannot find {A_SLUG} -- copy one of the paths above into A_SLUG")
print("A output root:", ROOT)

# locate processed/ and the manifests, however deeply they are nested
proc = ROOT / "processed"
if not proc.exists():
    hits = [p for p in ROOT.rglob("processed") if p.is_dir()]
    if not hits:
        sys.exit(f"no processed/ folder under {ROOT} -- wrong notebook version?")
    proc = hits[0]
src_man = proc.parent / "manifests"
if not src_man.exists():
    hits = list(ROOT.rglob("train_asr.jsonl"))
    if not hits:
        sys.exit(f"no manifests under {ROOT}")
    src_man = hits[0].parent
print("processed/:", proc)
print("manifests/:", src_man)


def repoint(old):
    """/kaggle/working/processed/test/chittagong/x.wav -> <proc>/test/chittagong/x.wav"""
    parts = Path(old).parts
    if "processed" in parts:
        tail = parts[parts.index("processed") + 1:]
        return str(proc.joinpath(*tail))
    return old


total_missing = 0
for f in ("train.jsonl", "val.jsonl", "test.jsonl",
          "train_asr.jsonl", "val_asr.jsonl"):
    p = src_man / f
    if not p.exists():
        print(f"  {f:<18} not present, skipped")
        continue
    rows, missing = [], 0
    for line in open(p, encoding="utf-8"):
        if not line.strip():
            continue
        r = json.loads(line)
        r["audio"] = repoint(r["audio"])
        if not Path(r["audio"]).exists():
            missing += 1
            if missing == 1:
                print(f"    first missing: {r['audio']}")
        rows.append(r)
    with open(DST / f, "w", encoding="utf-8") as out:
        for r in rows:
            out.write(json.dumps(r, ensure_ascii=False) + "\n")
    hrs = sum(r["duration"] for r in rows) / 3600
    extra = sum(1 for r in rows if r.get("dialect") == "_extra")
    flag = f"  MISSING AUDIO: {missing}" if missing else ""
    print(f"  {f:<18}{len(rows):>7} clips {hrs:>6.1f} h  extra:{extra}{flag}")
    total_missing += missing

print()
if total_missing:
    print(f"{total_missing} clips point at files that do not exist.")
    print("Check the 'first missing' path above against what is really on disk:")
    print(f"  !ls {proc}")
else:
    print("all audio paths resolve -- notebook C is ready")


Overwriting /kaggle/working/link_a.py


In [3]:
%%writefile /kaggle/working/w_train.py
"""Whisper-small (Bangla-adapted checkpoint) + LoRA, on train_asr.jsonl."""
import sys, json
from pathlib import Path

sys.path.insert(0, "/kaggle/working")
from asr_common import MANIFEST_DIR, CKPT_DIR, SR, read_manifest

import warnings, logging, inspect
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

import numpy as np
import torch
import soundfile as sf
from torch.utils.data import Dataset
from transformers import (WhisperProcessor, WhisperForConditionalGeneration,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer, set_seed)
from peft import LoraConfig, get_peft_model

MODEL_ID = "ashrafulparan/whisper-small-bangla"
EPOCHS = 5
BATCH = 4
GRAD_ACCUM = 8
LR = 3e-4
WARMUP = 30
MAX_LABEL_TOKENS = 448      
RESUME = None               

OUT = CKPT_DIR / "whisper"
LANG, TASK = "bengali", "transcribe"

print("\n" + "=" * 52)
print("CONFIG -- check before it runs for hours")
for _k in ("MODEL_ID", "EPOCHS", "BATCH", "GRAD_ACCUM", "LR", "WARMUP", "RESUME"):
    print(f"  {_k:<18}{globals()[_k]}")
print("=" * 52)

assert torch.cuda.is_available(), "GPU accelerator is off"
print("GPU:", torch.cuda.get_device_name(0))
set_seed(42)

processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANG, task=TASK)


class ASRDataset(Dataset):
    def __init__(self, manifest):
        rows = [r for r in read_manifest(manifest) if r.get("text")]
        keep, dropped = [], 0
        for r in rows:
            if len(processor.tokenizer(r["text"]).input_ids) <= MAX_LABEL_TOKENS:
                keep.append(r)
            else:
                dropped += 1
        self.rows = keep
        print(f"{Path(manifest).name}: {len(keep)} clips "
              f"({dropped} dropped, >{MAX_LABEL_TOKENS} tokens)")
        if not keep:
            raise SystemExit("no usable transcripts")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        y, _ = sf.read(r["audio"], dtype="float32")
        if y.ndim > 1:
            y = y.mean(axis=1)
        feats = processor.feature_extractor(y, sampling_rate=SR).input_features[0]
        return {"input_features": feats,
                "labels": processor.tokenizer(r["text"]).input_ids}


class Collator:
    def __call__(self, batch):
        f = processor.feature_extractor.pad(
            [{"input_features": b["input_features"]} for b in batch],
            return_tensors="pt")
        lab = processor.tokenizer.pad(
            [{"input_ids": b["labels"]} for b in batch], return_tensors="pt")
        ids = lab["input_ids"].masked_fill(lab.attention_mask.ne(1), -100)
        if (ids[:, 0] == processor.tokenizer.bos_token_id).all():
            ids = ids[:, 1:]
        f["labels"] = ids
        return f


train_ds = ASRDataset(MANIFEST_DIR / "train_asr.jsonl")
val_ds = ASRDataset(MANIFEST_DIR / "val_asr.jsonl")

model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.config.use_cache = False
model.generation_config.language = LANG
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None




model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "v_proj"]))
model.print_trainable_parameters()

args = Seq2SeqTrainingArguments(
    output_dir=str(OUT), per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, warmup_steps=WARMUP, num_train_epochs=EPOCHS,
    fp16=True, eval_strategy="epoch", save_strategy="epoch",
    logging_steps=25, save_total_limit=2, report_to=[],
    remove_unused_columns=False, label_names=["labels"],
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    greater_is_better=False)

kw = dict(model=model, args=args, train_dataset=train_ds,
          eval_dataset=val_ds, data_collator=Collator())


sig = inspect.signature(Seq2SeqTrainer.__init__).parameters
kw["processing_class" if "processing_class" in sig else "tokenizer"] = processor

trainer = Seq2SeqTrainer(**kw)
print(f"\nsteps/epoch ~ {len(train_ds) // (BATCH * GRAD_ACCUM)}")
trainer.train(resume_from_checkpoint=RESUME)

model.save_pretrained(str(OUT / "final"))
processor.save_pretrained(str(OUT / "final"))
print("\nsaved:", OUT / "final")


Overwriting /kaggle/working/w_train.py


In [4]:
%%writefile /kaggle/working/w_eval.py
"""Whisper on the 6-dialect test set."""
import os, sys, json
from pathlib import Path

sys.path.insert(0, "/kaggle/working")
from asr_common import MANIFEST_DIR, CKPT_DIR, SR, read_manifest, score, worst

import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

import numpy as np
import torch
import soundfile as sf
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_ID = "ashrafulparan/whisper-small-bangla"
ADAPTER = os.environ.get("ASR_ADAPTER", "").strip() or None
TAG = os.environ.get("ASR_TAG", "whisper_zeroshot").strip()
NUM_BEAMS = int(os.environ.get("ASR_BEAMS", "5"))
BATCH = 4 if NUM_BEAMS > 1 else 8      

LANG, TASK = "bengali", "transcribe"
GEN_KW = dict(max_length=448, language=LANG, task=TASK, num_beams=NUM_BEAMS)

print("=" * 52)
print(f"  adapter   {ADAPTER or 'none (zero-shot)'}")
print(f"  tag       {TAG}")
print(f"  beams     {NUM_BEAMS}" + ("  (greedy)" if NUM_BEAMS == 1 else "  (beam search)"))
print(f"  batch     {BATCH}")
print("=" * 52)

assert torch.cuda.is_available(), "GPU accelerator is off"
rows = [r for r in read_manifest(MANIFEST_DIR / "test.jsonl") if r.get("text")]
print(f"{len(rows)} test clips")

model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).half().cuda()
processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANG, task=TASK)
if ADAPTER:
    from peft import PeftModel
    assert Path(ADAPTER).exists(), f"no adapter at {ADAPTER}"
    model = PeftModel.from_pretrained(model, ADAPTER).half().cuda()
model.eval()
model.generation_config.forced_decoder_ids = None

hyps = []
for i in range(0, len(rows), BATCH):
    chunk = rows[i:i + BATCH]
    wavs = []
    for r in chunk:
        y, _ = sf.read(r["audio"], dtype="float32")
        wavs.append(y.mean(axis=1) if y.ndim > 1 else y)
    feats = processor.feature_extractor(
        wavs, sampling_rate=SR, return_tensors="pt").input_features.half().cuda()
    with torch.no_grad():
        out = model.generate(feats, **GEN_KW)
    hyps += processor.batch_decode(out, skip_special_tokens=True)
    if (i // BATCH) % 20 == 0:
        print(f"  {i + len(chunk)}/{len(rows)}")

score(rows, hyps, TAG, extra={"model": MODEL_ID, "adapter": str(ADAPTER),
                              "num_beams": NUM_BEAMS})
worst(TAG)


Overwriting /kaggle/working/w_eval.py


In [5]:
%%writefile /kaggle/working/n_manifest.py
"""train_asr.jsonl / val_asr.jsonl -> NeMo manifests."""
import json, sys, collections
from pathlib import Path

MANIFEST_DIR = Path("/kaggle/working/manifests")
NEMO_DIR = MANIFEST_DIR / "nemo"
NEMO_DIR.mkdir(parents=True, exist_ok=True)

MIN_DUR, MAX_DUR = 0.5, 20.0


def convert(src, dst):
    rows, skipped = [], collections.Counter()
    for line in open(src, encoding="utf-8"):
        if not line.strip():
            continue
        r = json.loads(line)
        txt = (r.get("text") or "").strip()
        dur = float(r["duration"])
        if not txt:
            skipped["no_text"] += 1; continue
        if dur < MIN_DUR:
            skipped["too_short"] += 1; continue
        if dur > MAX_DUR:
            skipped["too_long"] += 1; continue
        if not Path(r["audio"]).exists():
            skipped["missing_file"] += 1; continue
        rows.append({"audio_filepath": r["audio"], "duration": dur, "text": txt,
                     "dialect": r.get("dialect", "_extra")})

    with open(dst, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    hrs = sum(r["duration"] for r in rows) / 3600
    by = collections.Counter(r["dialect"] for r in rows)
    print(f"\n{dst.name}: {len(rows)} clips, {hrs:.2f} h")
    print("  by dialect:", dict(by))
    if skipped:
        print("  skipped:", dict(skipped))
    return rows


print("=" * 66); print("1. CONVERT"); print("=" * 66)
train = convert(MANIFEST_DIR / "train_asr.jsonl", NEMO_DIR / "train.json")
val = convert(MANIFEST_DIR / "val_asr.jsonl", NEMO_DIR / "val.json")
test = convert(MANIFEST_DIR / "test.jsonl", NEMO_DIR / "test.json")

print("\n" + "=" * 66); print("2. DID A3 ACTUALLY USE THE EXTRA REGIONS?"); print("=" * 66)
n_extra = sum(1 for r in train if r["dialect"] == "_extra")
print(f"  train clips           : {len(train)}")
print(f"  from extra regions    : {n_extra}")
if n_extra == 0:
    print("  >>> ZERO extra clips. You ran an OLD notebook A.")
    print("  >>> Set USE_ASR_EXTRA_REGIONS=True in common.py and re-run A3,")
    print("  >>> or you are training on the same ~5k clips as before.")
elif len(train) < 12000:
    print(f"  >>> Only {len(train)} clips; expected ~15,700. Check A3's output.")
else:
    print("  >>> Looks right.")

print("\n" + "=" * 66); print("3. TOKENIZER COVERAGE"); print("=" * 66)
print("The pretrained SentencePiece vocab was built on someone else's Bangla.")
print("If our dialectal text hits <unk> a lot, the model literally cannot")
print("learn to emit it and WER has a hard floor.\n")
try:
    import nemo.collections.asr as nemo_asr
    m = nemo_asr.models.ASRModel.from_pretrained(
        "SayedShaun/stt_bn_fastconformer_hybrid_large_v2", map_location="cpu")
    tok = m.tokenizer
    unk, tot, bad = 0, 0, collections.Counter()
    sample = train[:3000]
    for r in sample:
        ids = tok.text_to_ids(r["text"])
        back = tok.ids_to_text(ids)
        for ch in set(r["text"]) - set(back):
            if ch.strip():
                bad[ch] += 1
        tot += len(ids)
        unk += sum(1 for i in ids if i == getattr(tok, "unk_id", -999))
    print(f"  sampled {len(sample)} clips, {tot} tokens")
    print(f"  unk tokens: {unk} ({100*unk/max(tot,1):.3f}%)")
    if bad:
        print(f"  chars lost in round-trip: {dict(bad.most_common(15))}")
    else:
        print("  every character survives a text -> ids -> text round trip")
    print("\n  <0.5% unk  -> reuse the vocab, just fine-tune (what we want)")
    print("  >2%   unk  -> call change_vocabulary() and retrain the head")
    del m
except Exception as e:
    print("  skipped (needs nemo + download):", e)

print("\n" + "=" * 66); print("DONE"); print("=" * 66)
print(f"  {NEMO_DIR}/train.json  {len(train)}")
print(f"  {NEMO_DIR}/val.json    {len(val)}")
print(f"  {NEMO_DIR}/test.json   {len(test)}")


Overwriting /kaggle/working/n_manifest.py


In [6]:
%%writefile /kaggle/working/n_train.py
"""Fine-tune SayedShaun/stt_bn_fastconformer_hybrid_large_v2 on our dialect data."""
import os, json, glob, math, sys
from pathlib import Path

import torch
from omegaconf import DictConfig, OmegaConf, open_dict
import nemo.collections.asr as nemo_asr
from nemo.core.classes.modelPT import ModelPT


_roots = {c.__module__.split(".")[0] for c in ModelPT.__mro__}
if "lightning" in _roots:
    import lightning.pytorch as pl
else:
    import pytorch_lightning as pl
print("lightning namespace:", pl.__name__, pl.__version__)

MODEL_ID = "SayedShaun/stt_bn_fastconformer_hybrid_large_v2"
NEMO_DIR = Path("/kaggle/working/manifests/nemo")
OUT = Path("/kaggle/working/ckpt/nemo_ft"); OUT.mkdir(parents=True, exist_ok=True)

EPOCHS = 6
BATCH = 8
ACCUM = 4                 
LR = 3e-4
WARMUP = 100
FREEZE_ENCODER_LAYERS = 4  
CTC_LOSS_WEIGHT = 0.3      
MAX_DUR = 20.0
SAVE_STEPS = 500
RESUME = None             

print("\n" + "=" * 52)
print("CONFIG -- check these before it runs for an hour")
for _k in ("EPOCHS", "BATCH", "ACCUM", "LR", "WARMUP",
           "FREEZE_ENCODER_LAYERS", "CTC_LOSS_WEIGHT", "RESUME"):
    print(f"  {_k:<24}{globals()[_k]}")
print("=" * 52)

assert torch.cuda.is_available(), "GPU accelerator is off"
print("GPU:", torch.cuda.get_device_name(0))
for f in ("train.json", "val.json"):
    p = NEMO_DIR / f
    assert p.exists(), f"missing {p} -- run the manifest cell (n_manifest.py) first"
    print(f"  {f}: {sum(1 for _ in open(p))} lines")

print("\nloading pretrained ...")
model = nemo_asr.models.ASRModel.from_pretrained(MODEL_ID, map_location="cpu")

with open_dict(model.cfg):
    model.cfg.aux_ctc.ctc_loss_weight = CTC_LOSS_WEIGHT
    # SpecAugment: more masking than pretraining, we have far less data
    if "spec_augment" in model.cfg and model.cfg.spec_augment is not None:
        model.cfg.spec_augment.freq_masks = 2
        model.cfg.spec_augment.time_masks = 10


for _a in ("wer", "ctc_wer", "_wer", "_ctc_wer"):
    _m = getattr(model, _a, None)
    if _m is not None and hasattr(_m, "log_prediction"):
        _m.log_prediction = False
from nemo.utils import logging as nemo_logging
nemo_logging.setLevel("WARNING")      

ds = dict(sample_rate=16000, num_workers=2, pin_memory=True,
          max_duration=MAX_DUR, min_duration=0.5, is_tarred=False,
          trim_silence=False, use_start_end_token=False)

model.setup_training_data(DictConfig(
    dict(manifest_filepath=str(NEMO_DIR / "train.json"),
         batch_size=BATCH, shuffle=True, **ds)))
model.setup_validation_data(DictConfig(
    dict(manifest_filepath=str(NEMO_DIR / "val.json"),
         batch_size=BATCH, shuffle=False, **ds)))

n_train = sum(1 for _ in open(NEMO_DIR / "train.json"))
steps_per_epoch = math.ceil(n_train / (BATCH * ACCUM))
total_steps = steps_per_epoch * EPOCHS
print(f"\n{steps_per_epoch} optimizer steps/epoch x {EPOCHS} = {total_steps}")

model.setup_optimization(DictConfig(dict(
    name="adamw", lr=LR, betas=[0.9, 0.98], weight_decay=1e-3,
    sched=dict(name="CosineAnnealing", warmup_steps=WARMUP,
               min_lr=1e-6, max_steps=total_steps))))

frozen = 0
if FREEZE_ENCODER_LAYERS:
    for p in model.encoder.pre_encode.parameters():
        p.requires_grad = False; frozen += p.numel()
    for layer in model.encoder.layers[:FREEZE_ENCODER_LAYERS]:
        for p in layer.parameters():
            p.requires_grad = False; frozen += p.numel()
tot = sum(p.numel() for p in model.parameters())
print(f"frozen {frozen/1e6:.1f}M / {tot/1e6:.1f}M params "
      f"({sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M trainable)")

ckpt = pl.callbacks.ModelCheckpoint(
    dirpath=str(OUT / "checkpoints"), filename="{step}", save_last=True,
    every_n_train_steps=SAVE_STEPS, save_top_k=2,
    monitor="val_wer", mode="min")

trainer = pl.Trainer(
    accelerator="gpu", devices=1, precision="16-mixed",
    max_epochs=EPOCHS, accumulate_grad_batches=ACCUM,
    gradient_clip_val=1.0, log_every_n_steps=50,
    val_check_interval=SAVE_STEPS, enable_progress_bar=True,
    callbacks=[ckpt, pl.callbacks.LearningRateMonitor("step")],
    default_root_dir=str(OUT), logger=pl.loggers.CSVLogger(str(OUT), name="logs"),
    enable_checkpointing=True, num_sanity_val_steps=1)

model.set_trainer(trainer)
model.change_decoding_strategy(decoder_type="ctc")

print("\n" + "=" * 66); print("TRAINING"); print("=" * 66)
trainer.fit(model, ckpt_path=RESUME)

final = OUT / "dialect_fastconformer.nemo"
model.save_to(str(final))
print("\nsaved:", final, f"({final.stat().st_size/1e6:.0f} MB)")
print("checkpoints:", sorted(os.path.basename(p) for p in
                             glob.glob(str(OUT / "checkpoints" / "*.ckpt"))))
print("metrics csv:", sorted(glob.glob(str(OUT / "logs" / "*" / "metrics.csv"))))


Overwriting /kaggle/working/n_train.py


In [7]:
%%writefile /kaggle/working/n_eval.py
"""FastConformer on the 6-dialect test set."""
import sys
from pathlib import Path

sys.path.insert(0, "/kaggle/working")
from asr_common import read_manifest, score, worst

import os, functools
os.environ["TQDM_DISABLE"] = "1"
import tqdm as _tqdm
_tqdm.tqdm.__init__ = functools.partialmethod(_tqdm.tqdm.__init__, disable=True)
try:
    import tqdm.auto as _tqdm_auto
    _tqdm_auto.tqdm.__init__ = functools.partialmethod(
        _tqdm_auto.tqdm.__init__, disable=True)
except Exception:
    pass

import warnings, logging
warnings.filterwarnings("ignore")

import torch
from omegaconf import OmegaConf
import nemo.collections.asr as nemo_asr
from nemo.utils import logging as nemo_logging
nemo_logging.setLevel("ERROR")      

NEMO_DIR = Path("/kaggle/working/manifests/nemo")
DEFAULT_CKPT = "/kaggle/working/ckpt/nemo_ft/dialect_fastconformer.nemo"
CKPT = os.environ.get("ASR_NEMO_CKPT", "").strip() or DEFAULT_CKPT
TAG = os.environ.get("ASR_TAG", "nemo_finetuned").strip()
NUM_BEAMS = int(os.environ.get("ASR_BEAMS", "1"))
BATCH = 8 if NUM_BEAMS > 1 else 16

print("=" * 52)
print(f"  checkpoint  {CKPT}")
print(f"  tag         {TAG}")
print(f"  beams       {NUM_BEAMS}" + ("  (greedy)" if NUM_BEAMS == 1 else "  (beam)"))
print("=" * 52)

rows = read_manifest(NEMO_DIR / "test.json")
rows = [dict(r, audio=r["audio_filepath"]) for r in rows]
print(f"{len(rows)} test clips")

if CKPT.endswith(".nemo"):
    assert Path(CKPT).exists(), f"missing {CKPT} -- run the fine-tune cell first"
    model = nemo_asr.models.ASRModel.restore_from(CKPT, map_location="cuda")
else:
    model = nemo_asr.models.ASRModel.from_pretrained(CKPT, map_location="cuda")
model.eval()

if NUM_BEAMS > 1:
    model.change_decoding_strategy(
        OmegaConf.create({"strategy": "beam",
                          "beam": {"beam_size": NUM_BEAMS,
                                   "return_best_hypothesis": True}}),
        decoder_type="rnnt")
else:
    model.change_decoding_strategy(decoder_type="rnnt")

paths = [r["audio_filepath"] for r in rows]
try:
    hyps = model.transcribe(paths, batch_size=BATCH, verbose=False)
except TypeError:          # older signatures have no verbose kwarg
    hyps = model.transcribe(paths, batch_size=BATCH)
hyps = [h.text if hasattr(h, "text") else h for h in hyps]

score(rows, hyps, TAG, extra={"ckpt": CKPT, "head": "rnnt", "num_beams": NUM_BEAMS})
worst(TAG)

Overwriting /kaggle/working/n_eval.py


In [8]:
%%writefile /kaggle/working/err_analysis.py
"""Where do the ASR errors actually come from?"""
import sys, json, collections, glob
from pathlib import Path

sys.path.insert(0, "/kaggle/working")
from asr_common import RESULTS, norm, edits

TAGS = ["whisper_finetuned", "nemo_finetuned", "routed_asr"]
NEAR = 1          # char edits at or below this = "near miss"


def align(ref, hyp):
    """Levenshtein with backtrace -> list of (op, ref_word, hyp_word)."""
    R, H = len(ref), len(hyp)
    d = [[0] * (H + 1) for _ in range(R + 1)]
    for i in range(R + 1):
        d[i][0] = i
    for j in range(H + 1):
        d[0][j] = j
    for i in range(1, R + 1):
        for j in range(1, H + 1):
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1,
                          d[i - 1][j - 1] + (ref[i - 1] != hyp[j - 1]))
    ops, i, j = [], R, H
    while i > 0 or j > 0:
        if i > 0 and j > 0 and d[i][j] == d[i - 1][j - 1] + (ref[i - 1] != hyp[j - 1]):
            ops.append(("=" if ref[i - 1] == hyp[j - 1] else "S",
                        ref[i - 1], hyp[j - 1])); i -= 1; j -= 1
        elif i > 0 and d[i][j] == d[i - 1][j] + 1:
            ops.append(("D", ref[i - 1], None)); i -= 1
        else:
            ops.append(("I", None, hyp[j - 1])); j -= 1
    return ops[::-1]


for tag in TAGS:
    p = RESULTS / f"{tag}.json"
    if not p.exists():
        print(f"{tag}: not found, skipped"); continue
    clips = json.load(open(p))["per_clip"]

    cnt = collections.Counter()
    subdist = collections.Counter()
    wordpairs = collections.Counter()
    charpairs = collections.Counter()
    ref_words = 0
    near_misses = 0

    for c in clips:
        r, h = norm(c["ref"]).split(), norm(c["hyp"]).split()
        ref_words += len(r)
        for op, rw, hw in align(r, h):
            cnt[op] += 1
            if op == "S":
                dist = edits(list(rw), list(hw))
                subdist[min(dist, 5)] += 1
                if dist <= NEAR:
                    near_misses += 1
                    wordpairs[(rw, hw)] += 1
                    for a, b in zip(rw, hw):
                        if a != b:
                            charpairs[(a, b)] += 1
                            break

    err = cnt["S"] + cnt["D"] + cnt["I"]
    wer = 100 * err / max(ref_words, 1)
    forgiven = 100 * (err - near_misses) / max(ref_words, 1)

    print("\n" + "=" * 62)
    print(f"{tag}   WER {wer:.2f}   ({ref_words} reference words)")
    print("=" * 62)
    print(f"  substitutions {cnt['S']:>6}  {100*cnt['S']/err:>5.1f}% of errors")
    print(f"  deletions     {cnt['D']:>6}  {100*cnt['D']/err:>5.1f}%")
    print(f"  insertions    {cnt['I']:>6}  {100*cnt['I']/err:>5.1f}%")

    print("\n  how wrong are the substituted words?")
    tot_s = max(cnt["S"], 1)
    for k in sorted(subdist):
        lab = f"{k}+" if k == 5 else str(k)
        print(f"    {lab} char edits: {subdist[k]:>6}  {100*subdist[k]/tot_s:>5.1f}% of subs")

    print(f"\n  WER if <={NEAR}-char substitutions were forgiven: "
          f"{forgiven:.2f}  (from {wer:.2f}, -{wer-forgiven:.2f})")
    print("  ^ headroom a spelling-normalisation step could reach, at most")

    print("\n  most common near-miss word confusions (ref -> hyp):")
    for (a, b), n in wordpairs.most_common(15):
        print(f"    {n:>4}  {a}  ->  {b}")

    print("\n  most common single-character confusions:")
    for (a, b), n in charpairs.most_common(15):
        print(f"    {n:>4}  {a} -> {b}   (U+{ord(a):04X} -> U+{ord(b):04X})")


Overwriting /kaggle/working/err_analysis.py


In [9]:
%%writefile /kaggle/working/rescore.py
"""Recompute WER/CER for every saved run using the current norm()"""
import sys, json, collections, glob
from pathlib import Path

sys.path.insert(0, "/kaggle/working")
from asr_common import RESULTS, DIALECTS, norm, edits, repetition_rate

files = sorted(glob.glob(str(RESULTS / "*.json")))
files = [f for f in files if Path(f).stem != "asr_table"]
if not files:
    sys.exit("no results/*.json yet")

for f in files:
    d = json.load(open(f))
    if "per_clip" not in d:
        continue
    agg = collections.defaultdict(lambda: [0, 0, 0, 0, 0])
    for r in d["per_clip"]:
        rr, hh = norm(r["ref"]), norm(r["hyp"])
        rw, hw = rr.split(), hh.split()
        e1, n1 = edits(rw, hw), max(len(rw), 1)
        e2, n2 = edits(list(rr), list(hh)), max(len(rr), 1)
        r["wer"], r["cer"] = round(100 * e1 / n1, 2), round(100 * e2 / n2, 2)
        for k in (r["dialect"], "ALL"):
            a = agg[k]; a[0] += e1; a[1] += n1; a[2] += e2; a[3] += n2; a[4] += 1

    new = {}
    for k, a in agg.items():
        new[k] = {"n": a[4], "wer": round(100 * a[0] / max(a[1], 1), 2),
                  "cer": round(100 * a[2] / max(a[3], 1), 2)}

    old = d.get("summary", {})
    ow, nw = old.get("ALL", {}).get("wer"), new["ALL"]["wer"]
    delta = f"{nw - ow:+.2f}" if ow is not None else "n/a"
    print(f"{d['tag']:<26} WER {ow} -> {nw}  ({delta})")

    d["superseded"] = {"summary": old, "note": "scored before NFC normalisation"}
    d["summary"] = new
    d["repetition_pct"] = round(repetition_rate([r["hyp"] for r in d["per_clip"]]), 2)
    json.dump(d, open(f, "w"), ensure_ascii=False, indent=1)

print("\nrescored", len(files), "runs -- now re-run the comparison cell")


Overwriting /kaggle/working/rescore.py


In [10]:
%%writefile /kaggle/working/compare.py
"""Side-by-side table + figure over whatever results/*.json exist."""
import sys, json
from pathlib import Path

sys.path.insert(0, "/kaggle/working")
from asr_common import RESULTS, DIALECTS

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

FIG = Path("/kaggle/working/figures"); FIG.mkdir(parents=True, exist_ok=True)
ORDER = ["whisper_zeroshot", "whisper_finetuned", "nemo_zeroshot", "nemo_finetuned"]
LABEL = {"whisper_zeroshot": "Whisper (zero-shot)",
         "whisper_finetuned": "Whisper + LoRA",
         "nemo_zeroshot": "FastConformer (zero-shot)",
         "nemo_finetuned": "FastConformer + FT (proposed)"}

runs = {}
for t in ORDER:
    p = RESULTS / f"{t}.json"
    if p.exists():
        runs[t] = json.load(open(p))
if not runs:
    raise SystemExit("no results yet")
print("found:", ", ".join(runs))

cols = DIALECTS + ["ALL"]
print(f"\n{'system':<32}" + "".join(f"{c[:8]:>10}" for c in cols))
print("-" * (32 + 10 * len(cols)))
for t, d in runs.items():
    line = f"{LABEL[t]:<32}"
    for c in cols:
        line += f"{d['summary'][c]['wer']:>10.1f}" if c in d["summary"] else f"{'-':>10}"
    print(line)

print(f"\n{'system':<32}{'WER':>8}{'CER':>8}{'repeat%':>10}")
for t, d in runs.items():
    s = d["summary"]["ALL"]
    print(f"{LABEL[t]:<32}{s['wer']:>8.1f}{s['cer']:>8.1f}{d['repetition_pct']:>10.1f}")

# ---- figure: grouped WER per dialect ----
fig, ax = plt.subplots(figsize=(10, 4.2))
w = 0.8 / len(runs)
x = np.arange(len(cols))
for i, (t, d) in enumerate(runs.items()):
    vals = [d["summary"].get(c, {}).get("wer", np.nan) for c in cols]
    b = ax.bar(x + i * w - 0.4 + w / 2, vals, w, label=LABEL[t])
    ax.bar_label(b, fmt="%.0f", fontsize=7, padding=1)
ax.set_xticks(x); ax.set_xticklabels([c[:10] for c in cols], rotation=20, ha="right")
ax.set_ylabel("WER (%)"); ax.set_title("ASR word error rate by dialect")
ax.legend(fontsize=8, frameon=False); ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=.25, lw=.6); ax.set_axisbelow(True)
fig.tight_layout(); fig.savefig(FIG / "asr_comparison.png", dpi=160)
print("\nwrote", FIG / "asr_comparison.png")

json.dump({t: d["summary"] for t, d in runs.items()},
          open(RESULTS / "asr_table.json", "w"), indent=1)
print("wrote", RESULTS / "asr_table.json")


Overwriting /kaggle/working/compare.py


In [11]:
%%writefile /kaggle/working/report.py
"""Outputs: results table (CSV + LaTeX), significance tests,
and a reproducibility record."""
import sys, json, glob, os, platform, itertools
from pathlib import Path

sys.path.insert(0, "/kaggle/working")
from asr_common import RESULTS, DIALECTS, paired_bootstrap

OUT = Path("/kaggle/working/report"); OUT.mkdir(parents=True, exist_ok=True)

LABEL = {
    "whisper_zeroshot":         "Whisper-small (Bangla), zero-shot",
    "whisper_finetuned":        "Whisper-small (Bangla) + LoRA",
    "whisper_zeroshot_greedy":  "Whisper-small (Bangla), zero-shot [greedy]",
    "whisper_finetuned_greedy": "Whisper-small (Bangla) + LoRA [greedy]",
    "nemo_zeroshot":            "FastConformer-hybrid (Bangla), zero-shot",
    "nemo_finetuned":           "FastConformer-hybrid (Bangla) + fine-tuning",
}
ORDER = list(LABEL)

runs = {}
for t in ORDER:
    p = RESULTS / f"{t}.json"
    if p.exists():
        runs[t] = json.load(open(p))
if not runs:
    sys.exit("no results yet")


def ci_of(d, key="ALL", metric="wer"):
    """Return (lo, hi) or None. Tolerates the older files with no CI stored."""
    c = d["summary"].get(key, {}).get(f"{metric}_ci95")
    if not c or len(c) != 2:
        return None
    try:
        lo, hi = float(c[0]), float(c[1])
    except (TypeError, ValueError):
        return None
    if lo != lo or hi != hi:          # NaN
        return None
    return lo, hi


HAVE_CI = any(ci_of(d) for d in runs.values())
if not HAVE_CI:
    print("NOTE: no confidence intervals stored in results/*.json.\n"
          "      Run fix_ci.py once to add them, then re-run this script.\n"
          "      The LaTeX caption below omits the CI claim until they exist.\n")

# ---------------------------------------------------------------- 1. the table
cols = DIALECTS + ["ALL"]
lines = ["system,decoding," + ",".join(f"{c}_wer" for c in cols) +
         ",ALL_wer_ci_lo,ALL_wer_ci_hi,ALL_cer,ALL_cer_ci_lo,ALL_cer_ci_hi,repetition_pct"]

print("=" * 100)
print(f"{'system':<46}{'dec':>6}" + "".join(f"{c[:7]:>8}" for c in cols))
print("=" * 100)
for t, d in runs.items():
    beams = d.get("extra", {}).get("num_beams", "?")
    dec = "beam" if isinstance(beams, int) and beams > 1 else "greedy"
    row = f"{LABEL[t]:<46}{dec:>6}"
    csv = [f'"{LABEL[t]}"', dec]
    for c in cols:
        v = d["summary"].get(c, {}).get("wer")
        row += f"{v:>8.1f}" if v is not None else f"{'-':>8}"
        csv.append("" if v is None else f"{v:.2f}")
    print(row)
    a = d["summary"]["ALL"]
    w, c_ = ci_of(d, "ALL", "wer"), ci_of(d, "ALL", "cer")
    csv += [f"{w[0]:.2f}" if w else "", f"{w[1]:.2f}" if w else "",
            f"{a['cer']:.2f}",
            f"{c_[0]:.2f}" if c_ else "", f"{c_[1]:.2f}" if c_ else "",
            str(d.get("repetition_pct", ""))]
    lines.append(",".join(csv))

print("\nheadline numbers" +
      (" with 95% bootstrap CI (1000 resamples of utterances)" if HAVE_CI else ""))
for t, d in runs.items():
    a = d["summary"]["ALL"]
    w = ci_of(d, "ALL", "wer")
    ci = f"  [{w[0]:.2f}, {w[1]:.2f}]" if w else "  [CI not stored]"
    print(f"  {LABEL[t]:<46} WER {a['wer']:>6.2f}{ci}   CER {a['cer']:>6.2f}")

(OUT / "results.csv").write_text("\n".join(lines) + "\n")

# ----------------- 2. LaTeX
caption = (r"Word error rate (\%) by dialect on the held-out test set "
           r"($n=1{,}387$ utterances, session-disjoint split). Lower is better.")
if HAVE_CI:
    caption += (r" The final column gives the 95\% bootstrap confidence interval "
                r"for the overall score, from 1{,}000 resamples of utterances.")

head = ["System", "Dec."] + [("Overall" if c == "ALL" else c.capitalize())
        for c in cols]
if HAVE_CI:
    head.append(r"95\% CI")
spec = "l l " + "r" * len(cols) + ("c" if HAVE_CI else "")

tex = [r"\begin{table*}[t]", r"\centering", r"\small",
       r"\caption{" + caption + "}",
       r"\label{tab:asr}",
       r"\begin{tabular}{" + spec + "}", r"\toprule",
       " & ".join(head) + r" \\",
       r"\midrule"]
for t, d in runs.items():
    beams = d.get("extra", {}).get("num_beams", "?")
    dec = "beam" if isinstance(beams, int) and beams > 1 else "greedy"
    vals = []
    for c in cols:
        v = d["summary"].get(c, {}).get("wer")
        vals.append("--" if v is None else f"{v:.1f}")
    if HAVE_CI:
        w = ci_of(d, "ALL", "wer")
        vals.append(f"[{w[0]:.1f}, {w[1]:.1f}]" if w else "--")
    tex.append(f"{LABEL[t]} & {dec} & " + " & ".join(vals) + r" \\")
tex += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]
(OUT / "results_table.tex").write_text("\n".join(tex) + "\n")

# per-dialect CI table, for the appendix
if HAVE_CI:
    t2 = [r"\begin{table}[t]", r"\centering", r"\small",
          r"\caption{Per-dialect word error rate with 95\% bootstrap confidence "
          r"intervals for the best system. Intervals widen as the per-dialect "
          r"test count falls.}",
          r"\label{tab:asr-ci}",
          r"\begin{tabular}{l r r c}", r"\toprule",
          r"Dialect & $n$ & WER & 95\% CI \\", r"\midrule"]
    best = min(runs, key=lambda k: runs[k]["summary"]["ALL"]["wer"])
    for c in cols:
        s = runs[best]["summary"].get(c)
        if not s:
            continue
        w = ci_of(runs[best], c, "wer")
        name = "Overall" if c == "ALL" else c.capitalize()
        t2.append(f"{name} & {s.get('n','--')} & {s['wer']:.1f} & "
                  + (f"[{w[0]:.1f}, {w[1]:.1f}]" if w else "--") + r" \\")
    t2 += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    (OUT / "results_table_ci.tex").write_text("\n".join(t2) + "\n")

# --------------------------- 3. tests
print("\n" + "=" * 78)
print("PAIRED BOOTSTRAP TESTS  (p = probability the first system is NOT better)")
print("=" * 78)
pairs = [("whisper_finetuned", "whisper_zeroshot", "does fine-tuning help Whisper?"),
         ("nemo_finetuned", "nemo_zeroshot", "does fine-tuning help FastConformer?"),
         ("whisper_finetuned", "whisper_finetuned_greedy", "does beam search help?"),
         ("whisper_finetuned", "nemo_finetuned", "Whisper vs FastConformer")]
tests = []
for a, b, q in pairs:
    if a not in runs or b not in runs:
        print(f"  {q:<42} (needs {a} and {b})"); continue
    A, B = runs[a]["per_clip"], runs[b]["per_clip"]
    if len(A) != len(B):
        print(f"  {q:<42} SKIPPED: different test sets"); continue
    if "w_err" not in A[0] or "w_err" not in B[0]:
        print(f"  {q:<42} SKIPPED: rerun after the scoring update"); continue
    p = paired_bootstrap([r["w_err"] for r in A], [r["w_ref"] for r in A],
                         [r["w_err"] for r in B], [r["w_ref"] for r in B])
    wa = runs[a]["summary"]["ALL"]["wer"]; wb = runs[b]["summary"]["ALL"]["wer"]
    verdict = "significant" if p < 0.05 else "NOT significant"
    pstr = "< 0.001" if p == 0 else f"{p}"
    print(f"  {q:<42} {wa:.2f} vs {wb:.2f}   p={pstr:<9} {verdict}")
    tests.append({"question": q, "a": a, "b": b, "wer_a": wa, "wer_b": wb,
                  "p": p, "p_reported": pstr, "significant": bool(p < 0.05)})
json.dump(tests, open(OUT / "significance.json", "w"), indent=1)

# ------------------------------ 4. repro
env = {"python": platform.python_version(), "platform": platform.platform()}
for m in ("torch", "transformers", "peft", "numpy", "nemo"):
    try:
        env[m] = __import__(m).__version__
    except Exception:
        env[m] = None
try:
    import torch
    env["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except Exception:
    pass
env["systems"] = {t: d.get("extra", {}) for t, d in runs.items()}
json.dump(env, open(OUT / "environment.json", "w"), indent=1)

print("\nwrote:")
for f in sorted(OUT.iterdir()):
    print("  ", f)

Overwriting /kaggle/working/report.py


In [12]:
%%writefile /kaggle/working/fix_ci.py
"""Add 95% bootstrap confidence intervals to every results/*.json"""
import sys, json, glob
from pathlib import Path

import numpy as np

RESULTS = Path("/kaggle/working/results")
N_BOOT = 1000
SEED = 1234


def boot_ci(err, ref, n_boot=N_BOOT, seed=SEED):
    """Corpus-level rate = sum(err)/sum(ref), resampling utterances."""
    err = np.asarray(err, dtype=float)
    ref = np.asarray(ref, dtype=float)
    n = len(err)
    if n == 0 or ref.sum() <= 0:
        return None
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(n_boot, n))
    num = err[idx].sum(axis=1)
    den = ref[idx].sum(axis=1)
    good = den > 0
    if not good.any():
        return None
    rates = 100.0 * num[good] / den[good]
    return [round(float(np.percentile(rates, 2.5)), 2),
            round(float(np.percentile(rates, 97.5)), 2)]


def patch(path):
    d = json.load(open(path))
    per = d.get("per_clip")
    if not per:
        return f"{Path(path).stem}: no per_clip -- skipped"
    need = ("w_err", "w_ref", "c_err", "c_ref")
    missing = [k for k in need if k not in per[0]]
    if missing:
        return f"{Path(path).stem}: per_clip lacks {missing} -- re-run the eval"

    changed = []
    # overall
    for metric, ekey, rkey in (("wer", "w_err", "w_ref"), ("cer", "c_err", "c_ref")):
        ci = boot_ci([r[ekey] for r in per], [r[rkey] for r in per])
        if ci:
            d["summary"]["ALL"][f"{metric}_ci95"] = ci
            changed.append(f"{metric} {ci}")
    # per dialect
    for dia in {r.get("dialect") for r in per if r.get("dialect")}:
        rows = [r for r in per if r.get("dialect") == dia]
        if dia not in d["summary"]:
            continue
        for metric, ekey, rkey in (("wer", "w_err", "w_ref"), ("cer", "c_err", "c_ref")):
            ci = boot_ci([r[ekey] for r in rows], [r[rkey] for r in rows])
            if ci:
                d["summary"][dia][f"{metric}_ci95"] = ci

    json.dump(d, open(path, "w"), ensure_ascii=False)
    return f"{Path(path).stem}: " + "  ".join(changed)


files = sorted(glob.glob(str(RESULTS / "*.json")))
if not files:
    sys.exit(f"no result files in {RESULTS}")
print(f"patching {len(files)} result files with {N_BOOT}-replicate bootstrap CIs\n")
for f in files:
    print("  " + patch(f))
print("\ndone -- now re-run report.py")

Overwriting /kaggle/working/fix_ci.py


## C2 — attach notebook A's data, then verify

In [13]:
%cd /kaggle/working
!sed -i 's|^A_SLUG = .*|A_SLUG = "/kaggle/input/notebooks/tasfiaisrat/notebook-a-data-prep"|' link_a.py
%run link_a.py

/kaggle/working
A output root: /kaggle/input/notebooks/tasfiaisrat/notebook-a-data-prep
processed/: /kaggle/input/notebooks/tasfiaisrat/notebook-a-data-prep/processed
manifests/: /kaggle/input/notebooks/tasfiaisrat/notebook-a-data-prep/manifests
  train.jsonl          6942 clips   30.1 h  extra:0
  val.jsonl            1281 clips    5.7 h  extra:0
  test.jsonl           1387 clips    6.0 h  extra:0
  train_asr.jsonl     15539 clips   69.4 h  extra:8597
  val_asr.jsonl        1281 clips    5.7 h  extra:0

all audio paths resolve -- notebook C is ready


In [14]:
import json, torch
from pathlib import Path
M = Path("/kaggle/working/manifests")
tot = {}
for f in ("train_asr.jsonl", "val_asr.jsonl", "test.jsonl"):
    p = M / f
    if not p.exists():
        print(f"MISSING {f}"); continue
    rows = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
    hrs = sum(r["duration"] for r in rows) / 3600
    ex = sum(1 for r in rows if r.get("dialect") == "_extra")
    tot[f] = len(rows)
    print(f"{f:<18}{len(rows):>7} clips  {hrs:>6.1f} h  extra-region: {ex}")
print("\nGPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "OFF")

train_asr.jsonl     15539 clips    69.4 h  extra-region: 8597
val_asr.jsonl        1281 clips     5.7 h  extra-region: 0
test.jsonl           1387 clips     6.0 h  extra-region: 0

GPU: Tesla T4


# C3 — Whisper-small + LoRA

In [15]:
!rm -rf /kaggle/working/ckpt/whisper

In [16]:
%cd /kaggle/working
%run w_train.py

/kaggle/working



CONFIG -- check before it runs for hours
  MODEL_ID          ashrafulparan/whisper-small-bangla
  EPOCHS            5
  BATCH             4
  GRAD_ACCUM        8
  LR                0.0003
  WARMUP            30
  RESUME            None
GPU: Tesla T4


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.44k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

trainable params: 1,769,472 || all params: 243,504,384 || trainable%: 0.7267

steps/epoch ~ 321


Epoch,Training Loss,Validation Loss
1,2.831082,0.422639
2,2.630139,0.414316
3,2.401188,0.413948
4,2.273012,0.415987
5,2.244996,0.416808



saved: /kaggle/working/ckpt/whisper/final


# C4 — all four Whisper evaluations

In [17]:
import os, gc, json, runpy
import torch

RUNS = [
    ("",                                   "whisper_zeroshot",         5),
    ("/kaggle/working/ckpt/whisper/final", "whisper_finetuned",        5),
    ("",                                   "whisper_zeroshot_greedy",  1),
    ("/kaggle/working/ckpt/whisper/final", "whisper_finetuned_greedy", 1),
]

os.chdir("/kaggle/working")
for adapter, tag, beams in RUNS:
    print("\n" + "#" * 60)
    print(f"# {tag}")
    print("#" * 60)
    os.environ["ASR_ADAPTER"] = adapter
    os.environ["ASR_TAG"] = tag
    os.environ["ASR_BEAMS"] = str(beams)
    try:
        runpy.run_path("/kaggle/working/w_eval.py", run_name="__main__")
    except Exception as e:
        print(f"!! {tag} FAILED: {type(e).__name__}: {e}")
    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "=" * 62)
print(f"{'system':<30}{'beams':>7}{'WER':>10}{'CER':>9}")
print("=" * 62)
for _, tag, _ in RUNS:
    p = f"/kaggle/working/results/{tag}.json"
    if not os.path.exists(p):
        print(f"{tag:<30}{'-':>7}{'missing':>10}"); continue
    d = json.load(open(p))
    s = d["summary"]["ALL"]
    print(f"{tag:<30}{d['extra']['num_beams']:>7}{s['wer']:>10.2f}{s['cer']:>9.2f}")


############################################################
# whisper_zeroshot
############################################################
  adapter   none (zero-shot)
  tag       whisper_zeroshot
  beams     5  (beam search)
  batch     4
1387 test clips


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

  4/1387
  84/1387
  164/1387
  244/1387
  324/1387
  404/1387
  484/1387
  564/1387
  644/1387
  724/1387
  804/1387
  884/1387
  964/1387
  1044/1387
  1124/1387
  1204/1387
  1284/1387
  1364/1387

whisper_zeroshot
dialect            n      WER          95% CI      CER
standard         107    49.55  [44.71, 54.33]    16.26
chittagong       150    39.87  [35.39, 44.34]    26.16
mymensingh       272    57.72  [53.53, 61.74]    37.87
barishal         200    56.32  [52.01, 60.97]    36.76
sylhet           625    58.92  [56.66, 61.49]    35.55
noakhali          33    80.62  [76.59, 84.54]    46.11
ALL             1387    56.52  [54.91, 58.29]    34.91
repetition collapse: 0.6% of outputs
wrote /kaggle/working/results/whisper_zeroshot.json

worst 5 clips (whisper_zeroshot):
  [mymensingh] WER 1900
    REF: সমস্যা কি?
    HYP: সমস্যা কি? না দিলে সমস্যা নাই। কাপড় লইয়া বাইজ্জামগা। বিসুম অইলো বন্ধুত
  [barishal] WER 1467
    REF: যদি না আইতে
    HYP: না, না যাতে নাই ভাইয়া বেবোচনি কইরে ঠিক আছে

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

  4/1387
  84/1387
  164/1387
  244/1387
  324/1387
  404/1387
  484/1387
  564/1387
  644/1387
  724/1387
  804/1387
  884/1387
  964/1387
  1044/1387
  1124/1387
  1204/1387
  1284/1387
  1364/1387

whisper_finetuned
dialect            n      WER          95% CI      CER
standard         107    42.81  [38.48, 47.25]    12.76
chittagong       150    41.46  [37.04, 45.69]    26.26
mymensingh       272    56.39  [52.78, 59.95]    36.17
barishal         200    53.97  [50.24, 58.16]    34.12
sylhet           625    56.26  [54.12, 58.56]    34.27
noakhali          33    76.19   [72.02, 80.3]    45.88
ALL             1387    54.51  [53.05, 56.12]    33.47
repetition collapse: 0.5% of outputs
wrote /kaggle/working/results/whisper_finetuned.json

worst 5 clips (whisper_finetuned):
  [mymensingh] WER 1850
    REF: সমস্যা কি?
    HYP: সমস্যা কি? না দিলে সমস্যা নাই। কাফুর লইয়া বাইজ্জায়ামগা। বিশুম অইলো ব
  [barishal] WER 1400
    REF: যদি না আইতে
    HYP: না যাতে নাই ভাইয়া ব্যবস্থায় কিরে ঠিক 

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

  8/1387
  168/1387
  328/1387
  488/1387
  648/1387
  808/1387
  968/1387
  1128/1387
  1288/1387

whisper_zeroshot_greedy
dialect            n      WER          95% CI      CER
standard         107    50.75   [45.8, 55.66]    16.34
chittagong       150    41.51  [36.82, 46.07]    27.23
mymensingh       272    59.47  [55.29, 63.66]    38.92
barishal         200    58.71   [53.8, 63.68]    38.17
sylhet           625    59.85  [57.44, 62.48]    36.11
noakhali          33    82.65  [78.22, 87.23]    47.70
ALL             1387    57.96  [56.32, 59.84]    35.77
repetition collapse: 0.6% of outputs
wrote /kaggle/working/results/whisper_zeroshot_greedy.json

worst 5 clips (whisper_zeroshot_greedy):
  [mymensingh] WER 2050
    REF: সমস্যা কি?
    HYP: সমস্যা কি? না দিলে সমস্যা নাই। কাপুড় লইয়া বাইজাইউম কা? ইসুম অইলো বন্ধু
  [barishal] WER 1500
    REF: যদি না আইতে
    HYP: না, না যাতে নাই ভাইয়া বেবোচনি কইরে ঠিক আছে। নাই এ করি না, এহন কিছু করা
  [barishal] WER 1233
    REF: প্যাটটা ফাইড্ডা যাইব

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

  8/1387
  168/1387
  328/1387
  488/1387
  648/1387
  808/1387
  968/1387
  1128/1387
  1288/1387

whisper_finetuned_greedy
dialect            n      WER          95% CI      CER
standard         107    45.53  [40.95, 50.14]    14.11
chittagong       150    43.88  [38.98, 49.35]    27.27
mymensingh       272    59.41  [55.42, 63.12]    38.10
barishal         200    57.45  [53.35, 61.75]    36.59
sylhet           625    57.61  [55.32, 59.91]    35.05
noakhali          33    80.70   [75.57, 85.8]    48.81
ALL             1387    56.76  [55.21, 58.43]    34.83
repetition collapse: 1.0% of outputs
wrote /kaggle/working/results/whisper_finetuned_greedy.json

worst 5 clips (whisper_finetuned_greedy):
  [mymensingh] WER 1800
    REF: সমস্যা কি?
    HYP: সমস্যা কি? না দিলে সমস্যা নাই। কাফুর লইয়া বাইজ্জায়াম কা? কিসু মিলো ব
  [barishal] WER 1367
    REF: যদি না আইতে
    HYP: না যাতে নাই ভাইয়া ব্যবস্থা কিরে ঠিক আছে। নাই এক করিলে, এহন কিছু করার 
  [barishal] WER 800
    REF: প্যাটটা ফাইড্ডা যা

**End of session 1 — Quick Save.** Then a fresh session, GPU on, starting at C0b.

# C5 — FastConformer manifests

In [ ]:
%cd /kaggle/working
%run n_manifest.py

# C6 — FastConformer fine-tune

In [ ]:
!rm -rf /kaggle/working/ckpt/nemo_ft

In [ ]:
%cd /kaggle/working
%run n_train.py

# C7 — FastConformer evaluation 

In [ ]:
%env ASR_NEMO_CKPT=/kaggle/working/ckpt/nemo_ft/dialect_fastconformer.nemo
%env ASR_TAG=nemo_finetuned
%env ASR_BEAMS=4
%cd /kaggle/working
%run n_eval.py

In [ ]:
%env ASR_NEMO_CKPT=SayedShaun/stt_bn_fastconformer_hybrid_large_v2
%env ASR_TAG=nemo_zeroshot
%env ASR_BEAMS=4
%cd /kaggle/working
%run n_eval.py

# C8 — rescore every run with the current `norm()`

In [ ]:
%cd /kaggle/working
%run rescore.py

# C8b — bootstrap confidence intervals

`rescore.py` rebuilds `summary` from `per_clip`, which drops any CIs stored
earlier. This adds them back.


In [ ]:
%cd /kaggle/working
%run fix_ci.py


# C9 — comparison figure

In [ ]:
%cd /kaggle/working
%run compare.py

In [ ]:
from IPython.display import Image, display
display(Image("/kaggle/working/figures/asr_comparison.png"))

# C10 — error analysis

In [ ]:
%cd /kaggle/working
%run err_analysis.py

# C11 — publication outputs

Writes to `/kaggle/working/report/`:

- `results.csv` — the full table
- `results_table.tex` — a LaTeX table ready to paste
- `significance.json` — paired bootstrap tests
- `environment.json` — package versions, GPU, per-system settings

In [ ]:
%cd /kaggle/working
%run report.py

In [ ]:
print(open("/kaggle/working/report/results_table.tex").read())

# C12 — verify, then Save Version

In [ ]:
import json, os, glob
print("RESULTS")
for p in sorted(glob.glob("/kaggle/working/results/*.json")):
    d = json.load(open(p))
    if "summary" not in d:
        continue
    s = d["summary"]["ALL"]
    ci = s.get("wer_ci95", ["?", "?"])
    print(f"  {os.path.basename(p)[:-5]:<28} WER {s['wer']:>6.2f} "
          f"[{ci[0]}, {ci[1]}]  beams={d.get('extra', {}).get('num_beams', '?')}")
print("\nCHECKPOINTS")
for p in sorted(glob.glob("/kaggle/working/ckpt/*")):
    print("  ", p)
print("\nDISK"); os.system("du -sh /kaggle/working")